# Latihan 7: One-Hot Encoding

**Nama:** daffa Abdiantoro  
**NIM:** A11.2024.15898  
**Jurusan:** Teknik Informatika  

---

### Keterangan Latihan:
Latihan ini bertujuan untuk mempraktikkan teknik **One-Hot Encoding** menggunakan dua metode populer:
1. **Pandas `get_dummies()`** - Sangat mudah digunakan untuk analisis data eksploratif, namun tidak menyimpan informasi kategori untuk deployment.
2. **Scikit-Learn `OneHotEncoder`** - Ideal untuk pipeline machine learning karena dapat merekam kategori data train dan menerapkannya secara konsisten ke data test.

Dataset yang digunakan: `creditApprovalUCI.csv` (Credit Approval dataset).

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

## 2. Load Dataset
Kita akan memuat file `creditApprovalUCI.csv` yang sudah melalui proses data cleaning awal.

In [ ]:
# Memuat dataset
df = pd.read_csv('creditApprovalUCI.csv')
print("Ukuran Dataset:", df.shape)
display(df.head())

## 3. Split Train & Test Set
Pemisahan data dilakukan sebelum proses encoding untuk menghindari kebocoran data (*data leakage*).

In [ ]:
# Memisahkan fitur dan target (A16 adalah label target)
X = df.drop(columns=['A16'])
y = df['A16']

# Split train & test set dengan rasio 70:30
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test:", X_test.shape)

## 4. One-Hot Encoding dengan Pandas `get_dummies()`
Kita akan mempraktikkan penerapan `pd.get_dummies()` pada satu kolom yaitu `A4` dengan parameter `drop_first=True` untuk menghindari jebakan dummy variable (*dummy variable trap*).

In [ ]:
# Mengecek nilai unik di kolom A4 sebelum encoding
print("Nilai unik di A4:", X_train['A4'].unique())

# Menerapkan pd.get_dummies
df_a4_encoded = pd.get_dummies(X_train['A4'], drop_first=True)
print("\nHasil Encoding Kolom A4 (get_dummies):")
display(df_a4_encoded.head())

## 5. One-Hot Encoding pada Semua Kolom Kategorik (Pandas)
Kita akan memilih semua kolom bertipe data objek (kategorik) dan mengodekannya secara bersamaan.

In [ ]:
# Menentukan daftar kolom kategorik
vars_categorical = ['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']

# Encoding kolom-kolom kategorik tersebut untuk set train dan test
X_train_pandas = pd.get_dummies(X_train[vars_categorical], drop_first=True)
X_test_pandas = pd.get_dummies(X_test[vars_categorical], drop_first=True)

print("Jumlah kolom setelah encoding (Pandas):", X_train_pandas.shape[1])
display(X_train_pandas.head())

## 6. One-Hot Encoding dengan Scikit-Learn `OneHotEncoder`
Sekarang kita menggunakan kelas `OneHotEncoder` dari scikit-learn. Metode ini lebih disukai untuk pipeline produksi.

In [ ]:
# Inisialisasi OneHotEncoder
# categories='auto' mendeteksi kategori otomatis
# drop='first' menjatuhkan kolom pertama untuk menghindari multikolinearitas
# sparse_output=False agar output berupa dense numpy array (gunakan sparse=False untuk versi sklearn lama)
try:
    encoder = OneHotEncoder(categories='auto', drop='first', sparse_output=False)
except TypeError:
    # Untuk versi scikit-learn lama
    encoder = OneHotEncoder(categories='auto', drop='first', sparse=False)

# Fitting encoder pada data train
encoder.fit(X_train[vars_categorical])

# Transform data train dan test
X_train_sklearn = encoder.transform(X_train[vars_categorical])
X_test_sklearn = encoder.transform(X_test[vars_categorical])

print("Tipe data output Scikit-Learn:", type(X_train_sklearn))
print("Ukuran data train ter-encode:", X_train_sklearn.shape)
print("Contoh 5 baris pertama:")
print(X_train_sklearn[:5])

## 7. Kesimpulan
- **Pandas `get_dummies()`** mengembalikan objek `DataFrame` lengkap dengan nama kolom yang mudah dipahami, tetapi rawan inkonsistensi jika data train dan test memiliki jumlah kategori unik yang berbeda.
- **Scikit-Learn `OneHotEncoder`** mengembalikan array `numpy`, mempertahankan struktur fitur yang konsisten antara train dan test set, dan sangat cocok untuk deployment model Machine Learning.